In [1]:
import os
import json
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ======================================================
# Paths
# ======================================================

MODEL_PATH = "/content/drive/MyDrive/5.SemanticMapping/ClimateBERT_Scope/best_scope_classifier"

LAYOUT_JSON = "/content/drive/MyDrive/5.SemanticMapping/0_layoutlmv3_layout.json"

OUTPUT_DIR = "/content/drive/MyDrive/5.SemanticMapping/SemanticMapping_Inference"

OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "scope_predictions.json"
)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "scope_predictions.csv"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_LENGTH = 256

Load model

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model.to(DEVICE)

model.eval()

print("Model loaded.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded.


Load Label Mapping

In [5]:
with open(
    os.path.join(
        MODEL_PATH,
        "label_mapping.json"
    ),
    "r",
    encoding="utf-8"
) as f:

    mapping = json.load(f)

label2id = mapping["label2id"]

id2label = {
    int(k):v
    for k,v in mapping["id2label"].items()
}

Prediction Function

In [6]:
def predict_scope(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    inputs = {
        k:v.to(DEVICE)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        probs = F.softmax(
            outputs.logits,
            dim=-1
        )[0]

    pred = torch.argmax(probs).item()

    return {

        "scope":id2label[pred],

        "scope_id":pred,

        "confidence":float(
            probs[pred]
        ),

        "probabilities":{

            id2label[i]:float(probs[i])

            for i in range(len(probs))

        }
    }

Load Layout Output

In [7]:
with open(
    LAYOUT_JSON,
    "r",
    encoding="utf-8"
) as f:

    layout = json.load(f)

print(len(layout))

2554


Extract Text Blocks

In [10]:
blocks = []

for page in layout.values():

    page_id = page["page"]

    for block_id, block in enumerate(page["blocks"]):

        if block["type"] not in ["text","figure"]:
            continue

        text = block["text"].strip()

        if len(text)==0:
            continue

        blocks.append({

            "page":page_id,

            "block_id":block_id,

            "label":block["type"],

            "bbox":block["bbox"],

            "text":text

        })

print(len(blocks))

13622


ClimateBERT Inference

In [11]:
results = []

for block in tqdm(blocks):

    pred = predict_scope(
        block["text"]
    )

    results.append({

        **block,

        **pred

    })

  0%|          | 0/13622 [00:00<?, ?it/s]

Convert to DataFrame

In [12]:
df = pd.DataFrame(results)

df.head()

,page,block_id,label,bbox,text,scope,scope_id,confidence,probabilities
0,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,0,figure,"[813, 210, 831, 222]",VALUE,Other,0,0.999930,"{'Other': 0.999929666519165, 'Scope 1': 1.5481..."
1,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,1,figure,"[851, 379, 960, 424]",Ggo0 APPRECIATE WHAT WE HAVE\nENKIRONMENTAL wh...,Other,0,0.999943,"{'Other': 0.9999430179595947, 'Scope 1': 1.845..."
2,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,2,figure,"[798, 612, 935, 656]",60 9400\nSOCIAL Ve'll be shoring,Other,0,0.999936,"{'Other': 0.9999357461929321, 'Scope 1': 1.256..."
3,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,3,text,"[595, 864, 734, 888]",COMPATIBLES WITH DEVICES,Other,0,0.999926,"{'Other': 0.9999256134033203, 'Scope 1': 1.002..."
4,bao_viet_holdings_2023_p003_jpg.rf.a7d217e31e6...,0,text,"[410, 261, 508, 287]",WITH BAOVIET,Other,0,0.999639,"{'Other': 0.9996392726898193, 'Scope 1': 2.497..."


Save CSV

In [13]:
df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print(OUTPUT_CSV)

/content/drive/MyDrive/5.SemanticMapping/SemanticMapping_Inference/scope_predictions.csv


Save JSON

In [14]:
with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )

print(OUTPUT_JSON)

/content/drive/MyDrive/5.SemanticMapping/SemanticMapping_Inference/scope_predictions.json


Statistics

In [15]:
print(df["scope"].value_counts())

print()

print(df["confidence"].describe())

scope
Other      11005
Scope 3     1848
Scope 1      545
Scope 2      224
Name: count, dtype: int64

count    13622.000000
mean         0.995932
std          0.033303
min          0.424704
25%          0.999870
50%          0.999928
75%          0.999938
max          0.999950
Name: confidence, dtype: float64


Test Samples

In [16]:
display(

    df[

        [

            "page",

            "block_id",

            "scope",

            "confidence",

            "text"

        ]

    ].head(20)

)

,page,block_id,scope,confidence,text
0,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,0,Other,0.999930,VALUE
1,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,1,Other,0.999943,Ggo0 APPRECIATE WHAT WE HAVE\nENKIRONMENTAL wh...
2,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,2,Other,0.999936,60 9400\nSOCIAL Ve'll be shoring
3,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,3,Other,0.999926,COMPATIBLES WITH DEVICES
4,bao_viet_holdings_2023_p003_jpg.rf.a7d217e31e6...,0,Other,0.999639,WITH BAOVIET
5,bao_viet_holdings_2023_p003_jpg.rf.a7d217e31e6...,1,Scope 3,0.999848,"In 2023, the ""storm"" of the Covid-19 pandemic\..."
6,bao_viet_holdings_2023_p004_jpg.rf.b26c1222f8a...,0,Other,0.999929,#MODERN TECHNOLOGY
7,bao_viet_holdings_2023_p004_jpg.rf.b26c1222f8a...,1,Other,0.999734,#INTERACTIVE REPORT\n# ENVIRONMENTALLY FRIENDLY
8,bao_viet_holdings_2023_p004_jpg.rf.b26c1222f8a...,2,Other,0.999937,"In 2023, Baoviet has made great progress in be..."
9,bao_viet_holdings_2023_p005_jpg.rf.b14200e7e73...,0,Other,0.999909,RECOGNIZED AND RATED ASA SUSTAINABLE BUSINESS
